In [17]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql.functions import *
import time
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
import pandas as pd

spark = SparkSession.builder.master('local[*]').getOrCreate()

sc = spark.sparkContext
spark

In [ ]:
df = spark.read.parquet("yellow_tripdata_2025-01.parquet")
# 3. Mostrar las 20 primeras líneas SIN LIMPIAR
# Usamos truncate=False para que no corte los nombres de las columnas ni los textos
df.show(20, truncate=False)

# 4. (Opcional) Mostrar el esquema para revisar los tipos de datos de las columnas
df.printSchema()

#Clean the data 

# 1. Contar filas iniciales
total_inicial = df.count()

# 2. Aplicar únicamente el filtro de limpieza especificado
df_clean = df.filter(
    (col("passenger_count") > 0) & 
    (col("trip_distance") > 0) & 
    (col("fare_amount") > 0) & 
    (col("tip_amount") >= 0) &
    (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime")) & # Bajada posterior a la subida
    (year(col("tpep_pickup_datetime")) == 2025) &                # Asegurar año correcto (2025)
    (month(col("tpep_pickup_datetime")) == 1)                    # Asegurar mes de enero
)

# 3. Contar filas tras la limpieza y mostrar resumen
total_limpio = df_clean.count()
descartados = total_inicial - total_limpio

print(f"Total registros cargados:  {total_inicial:,}")
print(f"Total registros limpios:    {total_limpio:,}")
print(f"Registros descartados:     {descartados:,} ({ (descartados/total_inicial)*100:.2f}% del total)\n")

# Mostrar las 20 primeras filas del DataFrame limpio
df_clean.show(20, truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-01-01 00:18:38 |2025-01-01 00:26:59  |1              |1.6          |1         |N                 |229         |237 

In [19]:
from builtins import round as python_round

print("=================================================================")
print("ESTUDIO 1 (RDD API): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA")
print("=================================================================\n")

# 1. Contar filas totales
total_rows = df_clean.count()

# 2. Medir tiempo de ejecución
start_time_rdd = time.time()

# 3. Función map para extraer (hora, (velocidad, 1))
def extract_speed(row):
    seconds = (row['tpep_dropoff_datetime'] - row['tpep_pickup_datetime']).total_seconds()
    if seconds > 0:
        speed = (row['trip_distance'] / seconds) * 3600.0
        return (row['tpep_pickup_datetime'].hour, (speed, 1))
    return None

# 4. Pipeline de transformaciones RDD
rdd_res = df_clean.rdd \
    .map(extract_speed) \
    .filter(lambda x: x is not None) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: x[0] / x[1]) \
    .sortByKey() \
    .map(lambda x: (x[0], python_round(x[1], 2)))

# 5. Convertir a DataFrame y mostrar
df_res_rdd = rdd_res.toDF(['Hour', 'AverageSpeed'])
df_res_rdd.show(24)

execution_time_rdd = time.time() - start_time_rdd

# 6. Métricas de rendimiento
processing_speed_rdd = total_rows / execution_time_rdd

print(f"--- Métricas de Rendimiento (RDD) ---")
print(f"Tiempo de ejecución:        {execution_time_rdd:.4f} segundos")
print(f"Cantidad de datos:          {total_rows:,} filas")
print(f"Velocidad de procesamiento:  {processing_speed_rdd:,.2f} filas/segundo")

ESTUDIO 1 (RDD API): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA

+----+------------+
|Hour|AverageSpeed|
+----+------------+
|   0|       15.27|
|   1|       14.92|
|   2|       14.01|
|   3|       15.19|
|   4|       18.94|
|   5|        20.0|
|   6|       17.19|
|   7|        12.9|
|   8|       10.99|
|   9|       11.08|
|  10|       10.89|
|  11|       10.33|
|  12|       10.44|
|  13|       10.88|
|  14|       10.34|
|  15|       10.44|
|  16|       10.38|
|  17|       10.01|
|  18|       10.31|
|  19|        11.7|
|  20|       12.89|
|  21|       13.39|
|  22|       13.25|
|  23|       14.68|
+----+------------+

--- Métricas de Rendimiento (RDD) ---
Tiempo de ejecución:        91.4682 segundos
Cantidad de datos:          2,815,592 filas
Velocidad de procesamiento:  30,782.19 filas/segundo


In [ ]:
print("=================================================================")
print("ESTUDIO 1 (SPARK SQL): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA")
print("=================================================================\n")

# 1. Registrar vista temporal
total_rows = df_clean.count()

df_clean.createOrReplaceTempView("v_taxi")

# 2. Tu consulta adaptada (Sencilla y correcta)
query = """
    SELECT 
        hour(tpep_pickup_datetime) AS Hour, 
        ROUND(AVG(trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0)), 2) AS AverageSpeed
    FROM v_taxi
    WHERE unix_timestamp(tpep_dropoff_datetime) > unix_timestamp(tpep_pickup_datetime)
    GROUP BY hour(tpep_pickup_datetime)
    ORDER BY Hour
    """

# 4. Medir tiempo de ejecución
start_time_sql = time.time()

df_res_sql = spark.sql(query)
df_res_sql.show(24)

execution_time_sql = time.time() - start_time_sql

# 5. Calcular velocidad de procesamiento
processing_speed_sql = total_rows / execution_time_sql

# 6. Imprimir métricas de rendimiento
print(f"--- Métricas de Rendimiento (Spark SQL) ---")
print(f"Tiempo de ejecución:        {execution_time_sql:.4f} segundos")
print(f"Cantidad de datos:          {total_rows:,} filas")
print(f"Velocidad de procesamiento:  {processing_speed_sql:,.2f} filas/segundo")

ESTUDIO 1 (SPARK SQL): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA

+----+------------+
|Hour|AverageSpeed|
+----+------------+
|   0|       15.27|
|   1|       14.92|
|   2|       14.01|
|   3|       15.19|
|   4|       18.93|
|   5|        20.0|
|   6|       17.19|
|   7|        12.9|
|   8|       10.99|
|   9|       11.08|
|  10|       10.89|
|  11|       10.33|
|  12|       10.44|
|  13|       10.88|
|  14|       10.34|
|  15|       10.44|
|  16|       10.38|
|  17|       10.01|
|  18|       10.31|
|  19|        11.7|
|  20|       12.89|
|  21|       13.39|
|  22|       13.25|
|  23|       14.68|
+----+------------+

--- Métricas de Rendimiento (Spark SQL) ---
Tiempo de ejecución:        1.3123 segundos
Cantidad de datos:          2,815,592 filas
Velocidad de procesamiento:  2,145,461.60 filas/segundo


In [21]:
print("=================================================================")
print("ESTUDIO 1 (DATAFRAMES API): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA")
print("=================================================================\n")

# 1. Contar filas totales procesadas
total_rows = df_clean.count()

# 2. Medir tiempo de ejecución
start_time_df = time.time()

# 3. Transformaciones simples
df_res_df = df_clean \
    .where(unix_timestamp("tpep_dropoff_datetime") > unix_timestamp("tpep_pickup_datetime")) \
    .groupBy(hour("tpep_pickup_datetime").alias("Hour")) \
    .agg(
        round(avg(df_clean.trip_distance / ((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600.0)), 2).alias("AverageSpeed")
    ) \
    .orderBy("Hour")

# 4. Mostrar resultados (dispara la ejecución)
df_res_df.show(24)

execution_time_df = time.time() - start_time_df

# 5. Métricas de rendimiento
processing_speed_df = total_rows / execution_time_df

print(f"--- Métricas de Rendimiento (DataFrames) ---")
print(f"Tiempo de ejecución:        {execution_time_df:.4f} segundos")
print(f"Cantidad de datos:          {total_rows:,} filas")
print(f"Velocidad de procesamiento:  {processing_speed_df:,.2f} filas/segundo")

ESTUDIO 1 (DATAFRAMES API): VELOCIDAD MEDIA SEGÚN LA HORA DEL DÍA

+----+------------+
|Hour|AverageSpeed|
+----+------------+
|   0|       15.27|
|   1|       14.92|
|   2|       14.01|
|   3|       15.19|
|   4|       18.94|
|   5|        20.0|
|   6|       17.19|
|   7|        12.9|
|   8|       10.99|
|   9|       11.08|
|  10|       10.89|
|  11|       10.33|
|  12|       10.44|
|  13|       10.88|
|  14|       10.34|
|  15|       10.44|
|  16|       10.38|
|  17|       10.01|
|  18|       10.31|
|  19|        11.7|
|  20|       12.89|
|  21|       13.39|
|  22|       13.25|
|  23|       14.68|
+----+------------+

--- Métricas de Rendimiento (DataFrames) ---
Tiempo de ejecución:        0.6578 segundos
Cantidad de datos:          2,815,592 filas
Velocidad de procesamiento:  4,280,459.20 filas/segundo
